# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 06 — XGBoost

---

### Purpose
Train and evaluate an XGBoost gradient boosting classifier on all four feature sets.
XGBoost is expected to be competitive or superior on structured, dense features.

### Objectives
1. Load all feature matrices
2. Train XGBoost on each feature set
3. Full evaluation metrics suite
4. Confusion matrices, classification reports, ROC curves
5. XGBoost native feature importance (Gain, Weight, Cover)
6. SHAP analysis
7. Save trained models
8. Prediction examples

### Notebook Outline
1. Imports
2. Configuration
3. Load TF-IDF Features
4. Load Character N-Gram Features
5. Load Stylometric Features
6. Load Embedding Features
7. Training
8. Evaluation
9. Confusion Matrix
10. Classification Report
11. ROC Curves
12. XGBoost Feature Importance
13. SHAP Analysis
14. Model Saving
15. Prediction Examples
16. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classifiers import XGBoostClassifier
from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.evaluation.evaluator import ModelEvaluator
from src.visualization.plots import (
    plot_confusion_matrix, plot_roc_curves, plot_feature_importance
)
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    display_metrics_table, print_section_header,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED = cfg['random_seed']
TEST_SIZE   = cfg['evaluation']['test_size']
VAL_SIZE    = cfg['evaluation']['val_size']
XGB_CFG     = cfg['xgboost']
FEAT_CFG    = cfg['features']

set_global_seed(RANDOM_SEED)

DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

print(f'XGBoost Config : {XGB_CFG}')

---

## 3–6. Load Feature Matrices

In [ ]:
# XGBoost works best with dense arrays — convert sparse matrices
def load_and_split(matrix_path_str, labels_path_str, to_dense=False):
    X, y = load_feature_matrix(
        PROJECT_ROOT / matrix_path_str,
        PROJECT_ROOT / labels_path_str,
    )
    if to_dense and sp.issparse(X):
        X = X.toarray()
    return train_test_val_split(X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED)

X_tr_tf, X_val_tf, X_te_tf, y_tr_tf, y_val_tf, y_te_tf = load_and_split(
    FEAT_CFG['tfidf']['fingerprint'], FEAT_CFG['labels']['fingerprint'], to_dense=True
)
X_tr_ch, X_val_ch, X_te_ch, y_tr_ch, y_val_ch, y_te_ch = load_and_split(
    FEAT_CFG['char']['fingerprint'], FEAT_CFG['labels']['fingerprint'], to_dense=True
)
X_tr_st, X_val_st, X_te_st, y_tr_st, y_val_st, y_te_st = load_and_split(
    FEAT_CFG['style']['fingerprint'], FEAT_CFG['labels']['fingerprint'], to_dense=True
)
classes = np.load(
    str(PROJECT_ROOT / 'data' / 'features' / 'tfidf' / 'classes_tfidf_fingerprint.npy'),
    allow_pickle=True,
)
EMB_DIR = PROJECT_ROOT / 'data' / 'features' / 'embedding'
X_emb   = np.load(str(EMB_DIR / 'emb_fingerprint.npz'))['embeddings']
y_emb   = np.load(str(EMB_DIR / 'labels_emb_fingerprint.npy'))
X_tr_em, X_val_em, X_te_em, y_tr_em, y_val_em, y_te_em = train_test_val_split(
    X_emb, y_emb, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)

print('All feature matrices loaded (dense).')

---

## 7. Training

In [ ]:
def run_xgboost(X_train, X_test, y_train, y_test, feature_set: str):
    """Train XGBoost and return (model, evaluator)."""
    model = XGBoostClassifier(cfg=XGB_CFG)
    model.fit(X_train, y_train, feature_set=feature_set)

    evaluator = ModelEvaluator('xgboost', feature_set, classes)
    evaluator.evaluate(
        estimator=model.model,
        X_test=X_test,
        y_test=y_test,
        train_time=model.train_time_,
    )
    return model, evaluator

In [ ]:
print_section_header('Training XGBoost — TF-IDF Features')
xgb_tfidf, eval_xgb_tfidf = run_xgboost(X_tr_tf, X_te_tf, y_tr_tf, y_te_tf, 'tfidf')

print_section_header('Training XGBoost — Char N-Gram Features')
xgb_char, eval_xgb_char = run_xgboost(X_tr_ch, X_te_ch, y_tr_ch, y_te_ch, 'char')

print_section_header('Training XGBoost — Stylometric Features')
xgb_style, eval_xgb_style = run_xgboost(X_tr_st, X_te_st, y_tr_st, y_te_st, 'style')

print_section_header('Training XGBoost — Embedding Features')
xgb_emb, eval_xgb_emb = run_xgboost(X_tr_em, X_te_em, y_tr_em, y_te_em, 'embedding')

print('\n✅ All XGBoost variants trained.')

---

## 8. Evaluation

In [ ]:
results_xgb = pd.DataFrame([
    eval_xgb_tfidf.to_series(),
    eval_xgb_char.to_series(),
    eval_xgb_style.to_series(),
    eval_xgb_emb.to_series(),
])

display_cols = ['model_name','feature_set','accuracy','precision_macro',
                'recall_macro','f1_macro','f1_weighted','roc_auc_macro',
                'train_time_s','pred_time_s','peak_memory_mb']
results_xgb[display_cols].style.highlight_max(
    subset=['accuracy','f1_macro'], color='lightgreen'
).format(precision=4)

---

## 9. Confusion Matrix

In [ ]:
for evaluator, label in [
    (eval_xgb_tfidf, 'TF-IDF'),
    (eval_xgb_char,  'Char N-Gram'),
    (eval_xgb_style, 'Stylometric'),
    (eval_xgb_emb,   'Embedding'),
]:
    cm = np.array(evaluator.results_['confusion_matrix'])
    out_path = DIR_FIGURES / f'xgb_cm_{evaluator.feature_set}.png'
    plot_confusion_matrix(
        cm=cm, class_names=list(classes),
        title=f'XGBoost — {label} — Confusion Matrix',
        out_path=out_path, normalize=True,
    )
    print(f'✅ {out_path.name}')

---

## 10. Classification Report

In [ ]:
best_xgb_eval = max(
    [eval_xgb_tfidf, eval_xgb_char, eval_xgb_style, eval_xgb_emb],
    key=lambda e: e.results_['f1_macro'],
)
print(f'Best XGBoost feature set: {best_xgb_eval.feature_set}')
print(best_xgb_eval.results_['classification_report'])
best_xgb_eval.save_classification_report(
    DIR_OUTPUTS / f'xgb_{best_xgb_eval.feature_set}_classification_report.txt'
)

---

## 11. ROC Curves

In [ ]:
if best_xgb_eval.results_.get('y_proba') is not None:
    roc_path = DIR_FIGURES / f'xgb_roc_{best_xgb_eval.feature_set}.png'
    plot_roc_curves(
        y_test=best_xgb_eval.results_['y_test'],
        y_proba=best_xgb_eval.results_['y_proba'],
        class_names=list(classes),
        title=f'XGBoost — {best_xgb_eval.feature_set} — ROC Curves',
        out_path=roc_path,
    )
    print(f'✅ {roc_path.name}')

---

## 12. XGBoost Feature Importance

In [ ]:
# ── XGBoost supports 3 importance types ───────────────────────────────────────
# 'weight'  = number of times a feature is used in a split
# 'gain'    = average gain per split using the feature (most informative)
# 'cover'   = average coverage of splits using the feature

from src.feature_engineering.stylometric_extractor import StylometricExtractor
from src.utils.helpers import load_yaml as _load_yaml

feat_cfg_raw = _load_yaml(PROJECT_ROOT / 'configs' / 'feature_engineering.yaml')
style_ext = StylometricExtractor(cfg=feat_cfg_raw.get('stylometric', {}))
style_feature_names = style_ext._build_feature_names()

for importance_type in ['gain', 'weight', 'cover']:
    scores_dict = xgb_style.model.get_booster().get_score(
        importance_type=importance_type
    )
    # Map feature indices to names
    scores = np.zeros(len(style_feature_names))
    for k, v in scores_dict.items():
        idx = int(k.replace('f', '')) if k.startswith('f') else 0
        if idx < len(scores):
            scores[idx] = v

    path = DIR_FIGURES / f'xgb_style_{importance_type}_importance.png'
    plot_feature_importance(
        importances=scores,
        feature_names=style_feature_names,
        title=f'XGBoost — Stylometric Feature Importance ({importance_type.title()})',
        out_path=path,
        top_n=20,
    )
    print(f'✅ {path.name}')

---

## 13. SHAP Analysis

In [ ]:
try:
    import shap

    n_shap = min(100, X_te_st.shape[0])
    explainer = shap.TreeExplainer(xgb_style.model)
    shap_values = explainer.shap_values(X_te_st[:n_shap])

    plt.figure()
    shap.summary_plot(
        shap_values, X_te_st[:n_shap],
        feature_names=style_feature_names,
        show=False,
    )
    shap_path = DIR_FIGURES / 'xgb_shap_summary_stylometric.png'
    plt.savefig(shap_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'✅ SHAP summary: {shap_path.name}')

except ImportError:
    print('SHAP not installed. Run: pip install shap')

---

## 14. Model Saving

In [ ]:
for model, suffix in [
    (xgb_tfidf, 'tfidf'),
    (xgb_char,  'char'),
    (xgb_style, 'style'),
    (xgb_emb,   'embedding'),
]:
    saved_path = model.save(DIR_MODELS / 'xgboost', suffix=suffix)
    print(f'✅ Saved: {saved_path.name}')

---

## 15. Prediction Examples

In [ ]:
best_xgb = {
    'tfidf':     (xgb_tfidf, X_te_tf, y_te_tf),
    'char':      (xgb_char,  X_te_ch, y_te_ch),
    'style':     (xgb_style, X_te_st, y_te_st),
    'embedding': (xgb_emb,   X_te_em, y_te_em),
}[best_xgb_eval.feature_set]

model, X_te, y_te = best_xgb
y_pred_s  = model.predict(X_te[:10])
y_proba_s = model.predict_proba(X_te[:10])

pd.DataFrame({
    'True Label':      [classes[i] for i in y_te[:10]],
    'Predicted Label': [classes[i] for i in y_pred_s],
    'Correct':         y_te[:10] == y_pred_s,
    'Max Confidence':  np.max(y_proba_s, axis=1).round(4),
})

---

## 16. Notebook Summary

### XGBoost Results Summary

| Feature Set | Macro F1 | Weighted F1 | ROC-AUC | Train Time |
|---|---|---|---|---|
| TF-IDF | *(populate)* | *(populate)* | *(populate)* | *(populate)* |
| Char N-Gram | *(populate)* | *(populate)* | *(populate)* | *(populate)* |
| Stylometric | *(populate)* | *(populate)* | *(populate)* | *(populate)* |
| Embeddings | *(populate)* | *(populate)* | *(populate)* | *(populate)* |

→ **Notebook 07**: Hyperparameter Tuning

---
*Fingerprint Project — XGBoost — Complete*